# Dressel no-drive validation of the single-qubit AoT derivative

This notebook compares the derivative of $Q_N=\epsilon^2\sum_n(1+z_n^2)$ obtained from the exact Dressel no-drive solution, tangent-state propagation, and a paired central finite difference. It also checks the mean against the independent terminal-state identity $Q_{\rm D}=-\ln(1-z_T^2)$.

## 1. Parameter convention

Write $s=T/\tau=\gamma T=N\epsilon^2$. In this no-drive problem $J=0$, so the phase-diagram ratio $g=\gamma/(4J)$ is not defined. The validated derivative is

$$\partial_{\ln\gamma}\bar Q=\partial_{\ln s}\bar Q=s\,\partial_s\bar Q.$$

When a fixed nonzero $J$ is restored, $\ln g=\ln\gamma-\ln(4J)$ and the same logarithmic derivative becomes $\partial_{\ln g}$. We therefore plot against $\ln s$ and never identify $s$ itself with $\ln g$.

In [ ]:
import json
import os
import sys
from datetime import datetime
from pathlib import Path
from time import perf_counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.integrate import quad

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from quantum_measurement.aot_single_qubit import (
    dressel_Q_from_final_z,
    dressel_no_drive_pdf,
    dressel_no_drive_reference,
    paired_log_gamma_derivative_validation,
    rademacher_noise,
    simulate_ensemble,
)

SMOKE = os.environ.get('AOT_DRESSEL_SMOKE', '0') == '1'
OUTPUT_DIR = (Path('/tmp') / 'aot_single_qubit_dressel_validation_smoke') if SMOKE else (ROOT / 'results' / 'aot_single_qubit_dressel_validation')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
T = 1.0
J = 0.0
S_VALUES = np.array([0.2, 1.0]) if SMOKE else np.array([0.1, 0.2, 0.4, 0.7, 1.0, 1.5, 2.5, 4.0])
N_STEPS = 200 if SMOKE else 4000
N_STEPS_COARSE = N_STEPS // 2
N_TRAJECTORIES = 128 if SMOKE else 8192
H_PRIMARY = 0.01
H_COMPARISON = 0.02
SEED = 20260822
N_BOOTSTRAP = 100 if SMOKE else 2000
QUADRATURE_ORDER = 128
NONINFERIORITY_MARGIN = 1.10
RUN_ID = f'n{N_STEPS}_traj{N_TRAJECTORIES}_h{H_PRIMARY:g}_hc{H_COMPARISON:g}_boot{N_BOOTSTRAP}_seed{SEED}'
CHECKPOINT_DIR = OUTPUT_DIR / 'checkpoints' / RUN_ID
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_CSV = CHECKPOINT_DIR / 'completed_points.csv'
PROGRESS_LOG = OUTPUT_DIR / 'progress.log'
RUN_CONFIG = {
    'T': T, 'J': J, 's_values': S_VALUES.tolist(), 'n_steps': N_STEPS,
    'n_steps_coarse': N_STEPS_COARSE, 'n_trajectories': N_TRAJECTORIES,
    'h_primary': H_PRIMARY, 'h_comparison': H_COMPARISON,
    'seed': SEED, 'n_bootstrap': N_BOOTSTRAP,
}
with (CHECKPOINT_DIR / 'run_config.json').open('w', encoding='utf-8') as handle:
    json.dump(RUN_CONFIG, handle, indent=2)

print({'smoke': SMOKE, **RUN_CONFIG, 'checkpoint_dir': str(CHECKPOINT_DIR)})

## 2. Analytical reference

For an equatorial initial state, even functions of Dressel's integrated signal can be averaged with $\Gamma=s+\sqrt{s}Z$, $Z\sim\mathcal N(0,1)$. Hence

$$\bar Q_{\rm D}=\mathbb E[2\ln\cosh\Gamma],\qquad \chi_{\rm D}=s\,\mathbb E[2\tanh\Gamma+\operatorname{sech}^2\Gamma].$$

The following checks compare 64- and 128-node Gauss–Hermite quadrature and independently integrate Dressel et al. Eq. 14.

In [ ]:
analytic_checks = []
for s in (0.2, 1.0, 4.0):
    ref64 = dressel_no_drive_reference(s, quadrature_order=64)
    ref128 = dressel_no_drive_reference(s, quadrature_order=QUADRATURE_ORDER)
    normalization = quad(lambda Q: dressel_no_drive_pdf(Q, s), 0.0, np.inf, epsabs=1e-10)[0]
    pdf_mean = quad(lambda Q: Q * dressel_no_drive_pdf(Q, s), 0.0, np.inf, epsabs=1e-10)[0]
    analytic_checks.append({
        's': s, 'pdf_normalization': normalization,
        'pdf_mean': pdf_mean, 'quadrature_mean': ref128.mean_Q,
        'quadrature_64_128_delta': abs(ref64.mean_Q - ref128.mean_Q),
        'chi_64_128_delta': abs(ref64.chi_Q - ref128.chi_Q),
    })
analytic_check_table = pd.DataFrame(analytic_checks)
display(analytic_check_table)
assert np.allclose(analytic_check_table['pdf_normalization'], 1.0, atol=3e-10, rtol=0.0)
assert np.allclose(analytic_check_table['pdf_mean'], analytic_check_table['quadrature_mean'], rtol=3e-9)

## 3. Simulation and statistical helpers

The central, plus, and minus trajectories at each resolution share exactly the same Rademacher noise. The coarse run uses a separate deterministic seed because Rademacher increments do not admit an exact two-to-one Brownian aggregation.

In [ ]:
def mean_sem(values):
    values = np.asarray(values, dtype=float)
    return float(np.mean(values)), float(np.std(values, ddof=1) / np.sqrt(values.size))


def report_progress(message):
    line = f'{datetime.now().isoformat(timespec="seconds")} {message}'
    print(line, flush=True)
    with PROGRESS_LOG.open('a', encoding='utf-8') as handle:
        handle.write(line + '\n')


def checkpoint_name(s):
    return f's_{s:.8g}'.replace('.', 'p') + '.npz'


def bootstrap_efficiency_interval(tangent, finite_difference, tangent_time, fd_time, rng):
    tangent = np.asarray(tangent, dtype=float)
    finite_difference = np.asarray(finite_difference, dtype=float)
    n = tangent.size
    ratios = np.empty(N_BOOTSTRAP)
    for index in range(N_BOOTSTRAP):
        sample = rng.integers(0, n, size=n)
        tangent_variance = np.var(tangent[sample], ddof=1)
        fd_variance = np.var(finite_difference[sample], ddof=1)
        ratios[index] = tangent_variance * tangent_time / (fd_variance * fd_time)
    point = np.var(tangent, ddof=1) * tangent_time / (np.var(finite_difference, ddof=1) * fd_time)
    low, high = np.quantile(ratios, [0.025, 0.975])
    return float(point), float(low), float(high)


def run_resolution(s, n_steps, seed):
    dt = T / n_steps
    gamma = s / T
    noise = rademacher_noise(N_TRAJECTORIES, n_steps, seed)

    started = perf_counter()
    central, _ = simulate_ensemble(gamma, noise, J=J, dt=dt, burn_in=0)
    tangent_time = perf_counter() - started

    started = perf_counter()
    primary, _ = paired_log_gamma_derivative_validation(
        gamma, H_PRIMARY, noise, J=J, dt=dt, central=central
    )
    fd_time = perf_counter() - started

    comparison, _ = paired_log_gamma_derivative_validation(
        gamma, H_COMPARISON, noise, J=J, dt=dt, central=central
    )
    return central, primary, comparison, tangent_time, fd_time

## 4. Main scan

The fine run supplies the plotted estimates. A half-resolution run estimates finite-step sensitivity. Runtime is measured separately for one tangent propagation and the pair of physical propagations required by the primary finite difference.

In [ ]:
rows = []
samples_by_s = {}
seed_sequence = np.random.SeedSequence(SEED)
point_seeds = seed_sequence.spawn(2 * len(S_VALUES))
existing_by_s = {}
if CHECKPOINT_CSV.exists():
    checkpoint_table = pd.read_csv(CHECKPOINT_CSV)
    existing_by_s = {float(record['s']): record for record in checkpoint_table.to_dict('records')}
report_progress(f'run {RUN_ID}: {len(existing_by_s)}/{len(S_VALUES)} points already checkpointed')
scan_started = perf_counter()
computed_this_run = 0

for index, s in enumerate(S_VALUES):
    s = float(s)
    sample_path = CHECKPOINT_DIR / checkpoint_name(s)
    if s in existing_by_s and sample_path.exists():
        rows.append(existing_by_s[s])
        with np.load(sample_path) as saved:
            samples_by_s[s] = {name: saved[name] for name in saved.files}
        report_progress(f's={s:g}: restored checkpoint ({index + 1}/{len(S_VALUES)})')
        continue

    point_started = perf_counter()
    report_progress(f's={s:g}: starting ({index + 1}/{len(S_VALUES)})')
    fine_seed = int(point_seeds[2 * index].generate_state(1)[0])
    coarse_seed = int(point_seeds[2 * index + 1].generate_state(1)[0])
    fine, fd_fine, fd_comparison, tangent_time, fd_time = run_resolution(s, N_STEPS, fine_seed)
    coarse, fd_coarse, _, _, _ = run_resolution(s, N_STEPS_COARSE, coarse_seed)
    reference = dressel_no_drive_reference(s, quadrature_order=QUADRATURE_ORDER)

    dressel_terminal = np.asarray(dressel_Q_from_final_z(fine.final_z))
    dressel_terminal_coarse = np.asarray(dressel_Q_from_final_z(coarse.final_z))
    clipping_count = int(np.count_nonzero(1.0 - fine.final_z**2 <= np.finfo(float).tiny))

    Q_mean, Q_sem = mean_sem(fine.Q)
    Q_coarse_mean, _ = mean_sem(coarse.Q)
    terminal_mean, terminal_sem = mean_sem(dressel_terminal)
    terminal_coarse_mean, _ = mean_sem(dressel_terminal_coarse)
    tangent_mean, tangent_sem = mean_sem(fd_fine.tangent_Q)
    tangent_coarse_mean, _ = mean_sem(fd_coarse.tangent_Q)
    fd_mean, fd_sem = mean_sem(fd_fine.finite_difference_Q)
    fd_coarse_mean, _ = mean_sem(fd_coarse.finite_difference_Q)
    fd_comparison_mean, _ = mean_sem(fd_comparison.finite_difference_Q)

    Q_disc = abs(Q_mean - Q_coarse_mean)
    terminal_disc = abs(terminal_mean - terminal_coarse_mean)
    tangent_disc = abs(tangent_mean - tangent_coarse_mean)
    fd_disc = abs(fd_mean - fd_coarse_mean)
    fd_h_error = abs(fd_mean - fd_comparison_mean)
    efficiency, efficiency_low, efficiency_high = bootstrap_efficiency_interval(
        fd_fine.tangent_Q, fd_fine.finite_difference_Q, tangent_time, fd_time,
        np.random.default_rng(SEED + 1 + index),
    )

    Q_pass = abs(Q_mean - reference.mean_Q) <= 2.0 * Q_sem + Q_disc
    terminal_pass = abs(terminal_mean - reference.mean_Q) <= 2.0 * terminal_sem + terminal_disc
    tangent_pass = abs(tangent_mean - reference.chi_Q) <= 2.0 * tangent_sem + tangent_disc
    fd_pass = abs(fd_mean - reference.chi_Q) <= 2.0 * fd_sem + fd_disc + fd_h_error
    accuracy_pass = bool(Q_pass and terminal_pass and tangent_pass and fd_pass and clipping_count == 0)
    noninferiority_pass = bool(efficiency_high <= NONINFERIORITY_MARGIN)

    row = {
        's': s, 'ln_s': np.log(s), 'epsilon': np.sqrt(s / N_STEPS),
        'analytic_Q': reference.mean_Q, 'Q_mean': Q_mean, 'Q_sem': Q_sem,
        'terminal_Q_mean': terminal_mean, 'terminal_Q_sem': terminal_sem,
        'analytic_chi': reference.chi_Q, 'tangent_chi': tangent_mean,
        'tangent_sem': tangent_sem, 'fd_chi': fd_mean, 'fd_sem': fd_sem,
        'Q_discretization': Q_disc, 'terminal_discretization': terminal_disc,
        'tangent_discretization': tangent_disc, 'fd_discretization': fd_disc,
        'fd_h_error': fd_h_error,
        'pathwise_rms_h001': np.sqrt(np.mean(fd_fine.Q_difference**2)),
        'pathwise_rms_h002': np.sqrt(np.mean(fd_comparison.Q_difference**2)),
        'tangent_time_s': tangent_time, 'fd_time_s': fd_time,
        'efficiency_ratio': efficiency, 'efficiency_low': efficiency_low,
        'efficiency_high': efficiency_high, 'clipping_count': clipping_count,
        'accuracy_pass': accuracy_pass, 'noninferiority_pass': noninferiority_pass,
    }
    rows.append(row)
    samples_by_s[float(s)] = {
        'terminal_Q': dressel_terminal, 'tangent': fd_fine.tangent_Q,
        'finite_difference': fd_fine.finite_difference_Q,
    }
    np.savez_compressed(sample_path, **samples_by_s[s])
    pd.DataFrame(rows).sort_values('s').to_csv(CHECKPOINT_CSV, index=False)
    pd.DataFrame(rows).sort_values('s').to_csv(OUTPUT_DIR / 'dressel_validation.partial.csv', index=False)
    computed_this_run += 1
    point_seconds = perf_counter() - point_started
    remaining_new = len(S_VALUES) - index - 1
    average_seconds = (perf_counter() - scan_started) / computed_this_run
    eta_minutes = remaining_new * average_seconds / 60.0
    report_progress(
        f's={s:g}: checkpointed in {point_seconds:.1f}s; accuracy={accuracy_pass}; '
        f'efficiency upper={efficiency_high:.3g}; estimated remaining={eta_minutes:.1f} min'
    )

results = pd.DataFrame(rows).sort_values('s').reset_index(drop=True)
results.to_csv(OUTPUT_DIR / 'dressel_validation.csv', index=False)
report_progress(f'run {RUN_ID}: scan complete; final CSV written')
display(results[['s', 'analytic_Q', 'Q_mean', 'terminal_Q_mean', 'analytic_chi', 'tangent_chi', 'fd_chi', 'accuracy_pass', 'efficiency_ratio', 'efficiency_high', 'noninferiority_pass']])

## 5. Mean AoT and susceptibility

The lower panels show signed residuals from the analytical solution. Error bars are 95% normal confidence intervals; the separate convergence plots expose finite-step and finite-$h$ changes.

In [ ]:
s_smooth = np.geomspace(S_VALUES.min(), S_VALUES.max(), 300)
reference_smooth = [dressel_no_drive_reference(s) for s in s_smooth]
analytic_Q_smooth = np.array([item.mean_Q for item in reference_smooth])
analytic_chi_smooth = np.array([item.chi_Q for item in reference_smooth])
x = results['ln_s'].to_numpy()

fig, axes = plt.subplots(2, 1, figsize=(8, 7), sharex=True, gridspec_kw={'height_ratios': [3, 1]})
axes[0].plot(np.log(s_smooth), analytic_Q_smooth, color='black', label='Dressel analytical')
axes[0].errorbar(x, results['Q_mean'], yerr=1.96 * results['Q_sem'], fmt='o', capsize=3, label=r'$\epsilon^2\sum(1+z^2)$')
axes[0].errorbar(x, results['terminal_Q_mean'], yerr=1.96 * results['terminal_Q_sem'], fmt='s', capsize=3, label=r'$-\ln(1-z_T^2)$')
axes[0].set_ylabel(r'$\langle Q\rangle$')
axes[0].legend()
axes[0].grid(alpha=0.25)
axes[1].axhline(0.0, color='black', linewidth=1)
axes[1].errorbar(x, results['Q_mean'] - results['analytic_Q'], yerr=1.96 * results['Q_sem'], fmt='o', capsize=3, label='AoT sum')
axes[1].errorbar(x, results['terminal_Q_mean'] - results['analytic_Q'], yerr=1.96 * results['terminal_Q_sem'], fmt='s', capsize=3, label='terminal')
axes[1].set_xlabel(r'$\ln s=\ln(T/\tau)$')
axes[1].set_ylabel('residual')
axes[1].grid(alpha=0.25)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'mean_aot_vs_ln_s.png', dpi=200)
plt.show()

fig, axes = plt.subplots(2, 1, figsize=(8, 7), sharex=True, gridspec_kw={'height_ratios': [3, 1]})
axes[0].plot(np.log(s_smooth), analytic_chi_smooth, color='black', label='Dressel analytical')
axes[0].errorbar(x, results['tangent_chi'], yerr=1.96 * results['tangent_sem'], fmt='o', capsize=3, label='tangent state')
axes[0].errorbar(x, results['fd_chi'], yerr=1.96 * results['fd_sem'], fmt='s', capsize=3, label='paired finite difference')
axes[0].set_ylabel(r'$\partial_{\ln\gamma}\langle Q\rangle$')
axes[0].legend()
axes[0].grid(alpha=0.25)
axes[1].axhline(0.0, color='black', linewidth=1)
axes[1].errorbar(x, results['tangent_chi'] - results['analytic_chi'], yerr=1.96 * results['tangent_sem'], fmt='o', capsize=3, label='tangent')
axes[1].errorbar(x, results['fd_chi'] - results['analytic_chi'], yerr=1.96 * results['fd_sem'], fmt='s', capsize=3, label='finite difference')
axes[1].set_xlabel(r'$\ln s=\ln(T/\tau)$')
axes[1].set_ylabel('residual')
axes[1].grid(alpha=0.25)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'susceptibility_vs_ln_s.png', dpi=200)
plt.show()

## 6. Distribution cross-check at $s=1$

In [ ]:
distribution_s = 1.0 if 1.0 in samples_by_s else float(S_VALUES[-1])
terminal_Q = samples_by_s[distribution_s]['terminal_Q']
upper = max(float(np.quantile(terminal_Q, 0.997)), 1.0)
Q_grid = np.linspace(max(upper / 10000.0, 1e-8), upper, 1000)
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.hist(terminal_Q, bins=70, density=True, alpha=0.5, label='SSE terminal-state samples')
ax.plot(Q_grid, dressel_no_drive_pdf(Q_grid, distribution_s), color='black', linewidth=2, label='Dressel Eq. 14')
ax.set_xlabel(r'$Q_{\rm D}$')
ax.set_ylabel('probability density')
ax.set_title(f'No-drive AoT distribution, s={distribution_s:g}')
ax.legend()
ax.grid(alpha=0.2)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'dressel_eq14_distribution.png', dpi=200)
plt.show()

## 7. Discretization, finite-difference step, and efficiency diagnostics

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].semilogy(x, np.maximum(results['Q_discretization'], 1e-16), 'o-', label='AoT mean')
axes[0].semilogy(x, np.maximum(results['terminal_discretization'], 1e-16), 's-', label='terminal mean')
axes[0].semilogy(x, np.maximum(results['tangent_discretization'], 1e-16), '^-', label='tangent derivative')
axes[0].semilogy(x, np.maximum(results['fd_discretization'], 1e-16), 'd-', label='FD derivative')
axes[0].set_xlabel(r'$\ln s$')
axes[0].set_ylabel(f'absolute change: N={N_STEPS_COARSE} to {N_STEPS}')
axes[0].legend()
axes[0].grid(alpha=0.25)
axes[1].semilogy(x, np.maximum(results['fd_h_error'], 1e-16), 'o-', label=r'mean FD: $h=.02$ to $.01$')
axes[1].semilogy(x, np.maximum(results['pathwise_rms_h001'], 1e-16), 's-', label=r'RMS tangent-FD, $h=.01$')
axes[1].semilogy(x, np.maximum(results['pathwise_rms_h002'], 1e-16), '^-', label=r'RMS tangent-FD, $h=.02$')
axes[1].set_xlabel(r'$\ln s$')
axes[1].set_ylabel('finite-difference diagnostic')
axes[1].legend()
axes[1].grid(alpha=0.25)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'convergence_diagnostics.png', dpi=200)
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].semilogy(x, results['tangent_sem'], 'o-', label='tangent SEM')
axes[0].semilogy(x, results['fd_sem'], 's-', label='finite-difference SEM')
axes[0].set_xlabel(r'$\ln s$')
axes[0].set_ylabel('standard error')
axes[0].legend()
axes[0].grid(alpha=0.25)
ratio = results['efficiency_ratio'].to_numpy()
ratio_low = results['efficiency_low'].to_numpy()
ratio_high = results['efficiency_high'].to_numpy()
axes[1].errorbar(x, ratio, yerr=np.vstack((ratio-ratio_low, ratio_high-ratio)), fmt='o-', capsize=3)
axes[1].axhline(1.0, color='black', linewidth=1, label='equal efficiency')
axes[1].axhline(NONINFERIORITY_MARGIN, color='tab:red', linestyle='--', label='1.10 margin')
axes[1].set_yscale('log')
axes[1].set_xlabel(r'$\ln s$')
axes[1].set_ylabel('cost-adjusted variance ratio')
axes[1].legend()
axes[1].grid(alpha=0.25)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'estimator_efficiency.png', dpi=200)
plt.show()

## 8. Verdict

Accuracy requires agreement with the analytical result after allowing for two standard errors and the observed discretization/finite-$h$ changes. Non-inferiority is stricter: the upper 95% bootstrap bound on the cost-adjusted variance ratio must not exceed 1.10 at any point.

In [ ]:
accuracy_pass = bool(results['accuracy_pass'].all())
noninferiority_pass = bool(results['noninferiority_pass'].all())
verdict = {
    'accuracy_pass': accuracy_pass,
    'noninferiority_pass': noninferiority_pass,
    'overall_pass': bool(accuracy_pass and noninferiority_pass),
    'criterion': 'upper 95% bootstrap efficiency ratio <= 1.10 at every s',
    'smoke_configuration': SMOKE,
}
with (OUTPUT_DIR / 'verdict.json').open('w', encoding='utf-8') as handle:
    json.dump(verdict, handle, indent=2)

print(json.dumps(verdict, indent=2))
if accuracy_pass and noninferiority_pass:
    print('VERDICT: the tangent method is accurate and non-inferior to paired finite differences.')
elif accuracy_pass:
    print('VERDICT: analytical accuracy passed, but cost-adjusted non-inferiority did not pass.')
else:
    print('VERDICT: analytical accuracy did not pass under the stated resolution and uncertainty criteria.')